# Nivel 2 — LangChain + Gemini + Telegram + Tools de Stock/Pedidos

Este notebook reimplementa el mismo caso de uso del flujo visual de N8n en una solución programática con **LangChain**, **Gemini**, **Telegram Bot** y herramientas para **stock, pedidos y facturas**.

## Objetivo
- Recibir mensajes desde Telegram.
- Clasificar intención con Gemini.
- Crear pedidos o consultar estado.
- Leer y actualizar las hojas **Stock**, **Pedidos** y **Facturas**.
- Responder al usuario por Telegram.

## Secrets requeridos
- `GOOGLE_API_KEY`
- `TELEGRAM_BOT_TOKEN`

## Archivo de datos
Este notebook usa `Registros.xlsx`, con las hojas **Pedidos**, **Stock** y **Facturas**.

In [71]:
!pip install -q -U langchain langchain-core langchain-google-genai python-telegram-bot pandas==2.2.2 openpyxl pydantic==2.12.3 scikit-fuzzy
print("✅ Dependencias instaladas")

✅ Dependencias instaladas


## 1. Cargar credenciales

En Google Colab, guarda las llaves en **Secrets**. Si ejecutas localmente, también puedes definirlas como variables de entorno.

In [72]:
import os

try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    TELEGRAM_BOT_TOKEN = userdata.get("TELEGRAM_BOT_TOKEN")
    print("✅ GOOGLE_API_KEY cargada desde Colab Secrets")
    print("✅ TELEGRAM_BOT_TOKEN cargado desde Colab Secrets")
except Exception:
    TELEGRAM_BOT_TOKEN = os.getenv("TELEGRAM_BOT_TOKEN", "")
    if os.getenv("GOOGLE_API_KEY"):
        print("✅ GOOGLE_API_KEY cargada desde variables de entorno")
    else:
        print("⚠️ Falta GOOGLE_API_KEY")
    if TELEGRAM_BOT_TOKEN:
        print("✅ TELEGRAM_BOT_TOKEN cargado desde variables de entorno")
    else:
        print("⚠️ Falta TELEGRAM_BOT_TOKEN")

✅ GOOGLE_API_KEY cargada desde Colab Secrets
✅ TELEGRAM_BOT_TOKEN cargado desde Colab Secrets


## 2. Configuración general

In [73]:
from pathlib import Path
import pandas as pd
from datetime import datetime
from uuid import uuid4

EXCEL_PATH = Path("Registros.xlsx")
TZ_LABEL = "America/Bogota"

assert EXCEL_PATH.exists(), f"No se encontró {EXCEL_PATH.resolve()}"
print(f"✅ Archivo encontrado: {EXCEL_PATH.resolve()}")

✅ Archivo encontrado: /content/Registros.xlsx


## 3. Cargar hojas y revisar estructura

El archivo entregado contiene una tabla `Pedidos`, 30 productos en `Stock` ; además una tabla `Factura` , `Stock` usa las columnas `producto_id`, `descripcion_producto`, `stock` y `precio_unitario`, mientras `Facturas` usa `factura_id`, `order_id`, `cliente`, `monto_total`, `fecha_factura` y `estado_factura`.

In [74]:
stock_df = pd.read_excel(EXCEL_PATH, sheet_name="Stock")
pedidos_df = pd.read_excel(EXCEL_PATH, sheet_name="Pedidos")
facturas_df = pd.read_excel(EXCEL_PATH, sheet_name="Facturas")

print("Stock:", stock_df.shape)
print("Pedidos:", pedidos_df.shape)
print("Facturas:", facturas_df.shape)

stock_df.head()

Stock: (30, 4)
Pedidos: (4, 11)
Facturas: (2, 6)


,producto_id,descripcion_producto,stock,precio_unitario
0,PROD-001,Caja de guantes industriales talla M,120,18000
1,PROD-002,Cinta de embalaje 48 mm x 100 m,85,9500
2,PROD-003,Rollo de etiqueta térmica 100x100,24,32000
3,PROD-004,Lector de código de barras inalámbrico,12,145000
4,PROD-005,Impresora térmica de etiquetas,6,420000


## 4. Funciones auxiliares para Excel

Estas funciones simulan las tools del agente sobre el archivo `Registros.xlsx`.

In [75]:
EXPECTED_PEDIDOS = [
    "order_id", "cliente", "chat_id", "producto_id", "descripcion_producto",
    "cantidad", "estado", "stock", "fecha_pedido", "fecha_despacho", "total"
]
EXPECTED_STOCK = ["producto_id", "descripcion_producto", "stock", "precio_unitario"]
EXPECTED_FACTURAS = ["factura_id", "order_id", "cliente", "monto_total", "fecha_factura", "estado_factura"]


def now_str():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def load_sheets():
    stock = pd.read_excel(EXCEL_PATH, sheet_name="Stock")
    pedidos = pd.read_excel(EXCEL_PATH, sheet_name="Pedidos")
    facturas = pd.read_excel(EXCEL_PATH, sheet_name="Facturas")

    for col in EXPECTED_PEDIDOS:
        if col not in pedidos.columns:
            pedidos[col] = ""
    for col in EXPECTED_STOCK:
        if col not in stock.columns:
            stock[col] = ""
    for col in EXPECTED_FACTURAS:
        if col not in facturas.columns:
            facturas[col] = ""

    return stock[EXPECTED_STOCK].copy(), pedidos[EXPECTED_PEDIDOS].copy(), facturas[EXPECTED_FACTURAS].copy()


def save_sheets(stock, pedidos, facturas):
    with pd.ExcelWriter(EXCEL_PATH, engine="openpyxl", mode="w") as writer:
        pedidos.to_excel(writer, sheet_name="Pedidos", index=False)
        stock.to_excel(writer, sheet_name="Stock", index=False)
        facturas.to_excel(writer, sheet_name="Facturas", index=False)

### 4.1 Lógica difusa para priorización de pedidos

Este bloque implementa la técnica racional del Nivel 3 usando lógica difusa.
La prioridad del pedido se calcula a partir de tres variables: stock disponible,
cantidad solicitada y valor total del pedido.

In [76]:
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl

stock_u = np.arange(0, 201, 1)
cantidad_u = np.arange(0, 21, 1)
valor_u = np.arange(0, 1000001, 1000)
prioridad_u = np.arange(0, 101, 1)

stock_var = ctrl.Antecedent(stock_u, 'stock')
cantidad_var = ctrl.Antecedent(cantidad_u, 'cantidad')
valor_var = ctrl.Antecedent(valor_u, 'valor')
prioridad_var = ctrl.Consequent(prioridad_u, 'prioridad')

stock_var['bajo'] = fuzz.trapmf(stock_var.universe, [0, 0, 10, 40])
stock_var['medio'] = fuzz.trimf(stock_var.universe, [20, 80, 140])
stock_var['alto'] = fuzz.trapmf(stock_var.universe, [100, 140, 200, 200])

cantidad_var['baja'] = fuzz.trapmf(cantidad_var.universe, [0, 0, 2, 5])
cantidad_var['media'] = fuzz.trimf(cantidad_var.universe, [3, 7, 12])
cantidad_var['alta'] = fuzz.trapmf(cantidad_var.universe, [9, 13, 20, 20])

valor_var['bajo'] = fuzz.trapmf(valor_var.universe, [0, 0, 50000, 180000])
valor_var['medio'] = fuzz.trimf(valor_var.universe, [120000, 300000, 550000])
valor_var['alto'] = fuzz.trapmf(valor_var.universe, [450000, 650000, 1000000, 1000000])

prioridad_var['baja'] = fuzz.trimf(prioridad_var.universe, [0, 0, 40])
prioridad_var['media'] = fuzz.trimf(prioridad_var.universe, [30, 50, 70])
prioridad_var['alta'] = fuzz.trimf(prioridad_var.universe, [60, 100, 100])

rule1 = ctrl.Rule(stock_var['alto'] & cantidad_var['baja'], prioridad_var['alta'])
rule2 = ctrl.Rule(stock_var['medio'] & valor_var['alto'], prioridad_var['alta'])
rule3 = ctrl.Rule(stock_var['bajo'] & cantidad_var['alta'], prioridad_var['baja'])
rule4 = ctrl.Rule(stock_var['bajo'] & valor_var['alto'], prioridad_var['media'])
rule5 = ctrl.Rule(cantidad_var['media'] & valor_var['medio'], prioridad_var['media'])
rule6 = ctrl.Rule(stock_var['alto'] & valor_var['alto'], prioridad_var['alta'])
rule7 = ctrl.Rule(stock_var['medio'] & cantidad_var['baja'], prioridad_var['alta'])
rule8 = ctrl.Rule(stock_var['alto'] & cantidad_var['alta'], prioridad_var['media'])
rule9 = ctrl.Rule(stock_var['bajo'] & valor_var['bajo'], prioridad_var['baja'])

priority_ctrl = ctrl.ControlSystem([
    rule1, rule2, rule3, rule4, rule5, rule6, rule7, rule8, rule9
])

def priority_label(score: float) -> str:
    if score < 40:
        return "BAJA"
    elif score < 70:
        return "MEDIA"
    return "ALTA"

def fuzzy_priority(stock: int, cantidad: int, valor_total: float) -> dict:
    sim = ctrl.ControlSystemSimulation(priority_ctrl)
    sim.input['stock'] = max(0, min(200, int(stock)))
    sim.input['cantidad'] = max(0, min(20, int(cantidad)))
    sim.input['valor'] = max(0, min(1000000, float(valor_total)))
    sim.compute()
    score = float(sim.output['prioridad'])
    return {
        "prioridad_score": round(score, 2),
        "prioridad_label": priority_label(score)
    }

print("✅ Sistema difuso cargado")

✅ Sistema difuso cargado


In [77]:
print(fuzzy_priority(stock=120, cantidad=2, valor_total=64000))
print(fuzzy_priority(stock=5, cantidad=1, valor_total=890000))
print(fuzzy_priority(stock=0, cantidad=4, valor_total=120000))

{'prioridad_score': 84.44, 'prioridad_label': 'ALTA'}
{'prioridad_score': 50.0, 'prioridad_label': 'MEDIA'}
{'prioridad_score': 15.85, 'prioridad_label': 'BAJA'}


## 5. Tools del caso logístico

El flujo N8n del proyecto hace exactamente estas operaciones: clasifica intención, registra pedido, consulta `Stock`, valida disponibilidad, actualiza `Pedidos`, genera `Facturas` y notifica por Telegram.

In [78]:
from typing import Optional, Literal
from pydantic import BaseModel, Field

class IntentOutput(BaseModel):
    intencion: Literal["crear_pedido", "consultar_estado", "saludo", "otro"]
    order_id: str = ""
    producto_id: str = ""
    cantidad: int = 0


def get_stock(producto_id: str) -> dict:
    stock_df, _, _ = load_sheets()
    row = stock_df[stock_df["producto_id"].astype(str).str.upper() == producto_id.upper()]
    if row.empty:
        return {"found": False, "producto_id": producto_id, "mensaje": "Producto no encontrado"}
    r = row.iloc[0]
    return {
        "found": True,
        "producto_id": str(r["producto_id"]),
        "descripcion_producto": str(r["descripcion_producto"]),
        "stock": int(r["stock"]),
        "precio_unitario": float(r["precio_unitario"]),
    }


def createorder(cliente: str, chat_id: str, producto_id: str, cantidad: int) -> dict:
    stock_df, pedidos, facturas = load_sheets() # Use stock_df for the DataFrame
    info = get_stock(producto_id)
    order_id = f"{cliente}{str(uuid4())[:8]}"
    fecha_pedido = now_str()

    if not info["found"]:
        newrow = {
            "orderid": order_id,
            "cliente": cliente,
            "chat_id": str(chat_id),
            "producto_id": producto_id,
            "descripcion_producto": "",
            "cantidad": int(cantidad),
            "estado": "PRODUCTO_NO_ENCONTRADO",
            "stock": 0,
            "fecha_pedido": fecha_pedido,
            "fecha_despacho": "",
            "total": 0,
        }
        pedidos = pd.concat([pedidos, pd.DataFrame([newrow])], ignore_index=True)
        save_sheets(stock_df, pedidos, facturas) # Pass stock_df
        return {
            "ok": False,
            "order_id": order_id,
            "estado": "PRODUCTO_NO_ENCONTRADO",
            "mensaje": "Producto no encontrado"
        }

    current_stock_count = int(info["stock"]) # Rename to avoid shadowing the DataFrame
    precio_unitario = float(info["precio_unitario"])
    total = int(cantidad * precio_unitario)

    if current_stock_count >= cantidad:
        stock_df.loc[ # Use stock_df here
            stock_df["producto_id"].astype(str).str.upper() == producto_id.upper(),
            "stock"
        ] = current_stock_count - cantidad # Update the DataFrame with the new stock count

        fecha_despacho = now_str()
        prioridad = fuzzy_priority(
            stock=current_stock_count, # Use current_stock_count
            cantidad=int(cantidad),
            valor_total=float(total)
        )

        newrow = {
            "order_id": order_id,
            "cliente": cliente,
            "chat_id": str(chat_id),
            "producto_id": producto_id,
            "descripcion_producto": info["descripcion_producto"],
            "cantidad": int(cantidad),
            "estado": "DESPACHADO",
            "stock": current_stock_count, # Use current_stock_count
            "fecha_pedido": fecha_pedido,
            "fecha_despacho": fecha_despacho,
            "total": total,
        }

        pedidos = pd.concat([pedidos, pd.DataFrame([newrow])], ignore_index=True)

        factura = {
            "factura_id": f"FAC-{order_id}",
            "order_id": order_id,
            "cliente": cliente,
            "monto_total": total,
            "fecha_factura": fecha_despacho,
            "estado_factura": "GENERADA",
        }

        facturas = pd.concat([facturas, pd.DataFrame([factura])], ignore_index=True)
        save_sheets(stock_df, pedidos, facturas) # Pass stock_df

        return {
            "ok": True,
            "order_id": order_id,
            "estado": "DESPACHADO",
            "producto_id": producto_id,
            "descripcion_producto": info["descripcion_producto"],
            "cantidad": int(cantidad),
            "stock": current_stock_count, # Use current_stock_count
            "precio_unitario": precio_unitario,
            "total": total,
            "fecha_pedido": fecha_pedido,
            "fecha_despacho": fecha_despacho,
            "prioridad_score": prioridad["prioridad_score"],
            "prioridad_label": prioridad["prioridad_label"],
        }

    newrow = {
        "order_id": order_id,
        "cliente": cliente,
        "chat_id": str(chat_id),
        "producto_id": producto_id,
        "descripcion_producto": info["descripcion_producto"],
        "cantidad": int(cantidad),
        "estado": "SIN_STOCK",
        "stock": current_stock_count, # Use current_stock_count
        "fecha_pedido": fecha_pedido,
        "fecha_despacho": now_str(),
        "total": 0,
    }

    pedidos = pd.concat([pedidos, pd.DataFrame([newrow])], ignore_index=True)
    save_sheets(stock_df, pedidos, facturas) # Pass stock_df

    prioridad = fuzzy_priority(
        stock=current_stock_count, # Use current_stock_count
        cantidad=int(cantidad),
        valor_total=float(total)
    )

    return {
        "ok": False,
        "order_id": order_id,
        "estado": "SIN_STOCK",
        "producto_id": producto_id,
        "descripcion_producto": info["descripcion_producto"],
        "cantidad": int(cantidad),
        "stock": current_stock_count, # Use current_stock_count
        "total": 0,
        "prioridad_score": prioridad["prioridad_score"],
        "prioridad_label": prioridad["prioridad_label"],
    }


def get_order_status(chat_id: str, order_id: Optional[str] = None) -> dict:
    _, pedidos, _ = load_sheets()
    pedidos["chat_id"] = pedidos["chat_id"].astype(str)
    rows = pedidos[pedidos["chat_id"] == str(chat_id)].copy()
    if order_id:
        rows = rows[rows["order_id"].astype(str) == str(order_id)]
    rows = rows[rows["estado"].astype(str) != "SIN_STOCK"]
    if rows.empty:
        return {"ok": False, "mensaje": "No encontré pedidos válidos para consultar el estado."}
    last = rows.iloc[-1].to_dict()
    return {"ok": True, **last}

## 6. Modelo Gemini + Prompt + salida estructurada

El flujo N8n usa Gemini con un parser estructurado que obliga a devolver `intencion`, `order_id`, `producto_id` y `cantidad`; esa misma idea se replica aquí con LangChain y Pydantic.

In [89]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-pro",
    temperature=0,
)

structured_llm = llm.with_structured_output(IntentOutput)

prompt = ChatPromptTemplate.from_messages([
    ("system", """
Eres un asistente de logística que recibe mensajes por Telegram.
Debes identificar la intención del usuario y devolver únicamente un objeto estructurado.

Reglas:
- intencion solo puede ser: crear_pedido, consultar_estado, saludo, otro.
- Si el usuario quiere comprar o pedir un producto, usa crear_pedido.
- Si el usuario pregunta por el estado de un pedido, usa consultar_estado.
- Si solo saluda, usa saludo.
- Si no aplica ninguna, usa otro.
- producto_id debe verse como PROD-XXX si aparece.
- cantidad debe ser numérica.
- Si falta información, deja el campo vacío o en 0.
"""),
    ("human", "Mensaje del usuario: {mensaje}")
])

classifier_chain = prompt | structured_llm

## 7. Lógica del asistente

In [90]:
def build_response(parsed: IntentOutput, cliente: str, chat_id: str) -> str:
    if parsed.intencion == "saludo":
        return (
            "Hola, soy tu asistente de logística. "
            "Puedo ayudarte a crear pedidos y consultar el estado de tus pedidos."
        )

    if parsed.intencion == "crear_pedido":
        if not parsed.producto_id or int(parsed.cantidad or 0) <= 0:
            return (
                "Para crear el pedido necesito un producto y una cantidad. "
                "Ejemplo: quiero pedir PROD-003 cantidad 2"
            )
        result = createorder(cliente=cliente, chat_id=chat_id, producto_id=parsed.producto_id, cantidad=int(parsed.cantidad))

        if result.get("estado") == "DESPACHADO":
            return (
                f"Tu pedido {result['order_id']} fue procesado correctamente.\n"
                f"Producto: {result['descripcion_producto']}\n"
                f"Cantidad: {result['cantidad']}\n"
                f"Total: {result['total']}\n"
                f"Estado: DESPACHADO\n"
                f"Prioridad logística: {result['prioridad_label']}\n"
                f"Puntaje difuso: {result['prioridad_score']}"
            )

        if result.get("estado") == "SIN_STOCK":
            return (
                f"Tu pedido {result['orderid']} no pudo procesarse.\n"
                f"Producto: {result['descripcion_producto']}\n"
                f"Cantidad solicitada: {result['cantidad']}\n"
                f"Stock: {result['stock']}\n"
                f"Estado: SIN_STOCK\n"
                f"Prioridad logística: {result['prioridad_label']}\n"
                f"Puntaje difuso: {result['prioridad_score']}"
            )

    if parsed.intencion == "consultar_estado":
        result = get_order_status(chat_id=chat_id, order_id=parsed.order_id or None)
        if not result.get("ok"):
            return result["mensaje"]
        return (
            f"""Pedido {result['order_id']}
Producto: {result.get('descripcion_producto') or result.get('producto_id')}
Cantidad: {result['cantidad']}
Estado: {result['estado']}
Total: {result.get('total', 0)}
Fecha pedido: {result.get('fecha_pedido', 'No registrada')}
Fecha despacho: {result.get('fecha_despacho', 'Pendiente')}"""
        )

    return (
        """No entendí tu solicitud. Puedes escribir algo como:
- quiero pedir PROD-002 cantidad 2
- consultar estado de mi pedido"""
    )


def process_message(mensaje: str, cliente: str = "Daniel", chat_id: str = "6172774306"):
    parsed = classifier_chain.invoke({"mensaje": mensaje})
    answer = build_response(parsed, cliente=cliente, chat_id=str(chat_id))
    return parsed, answer


In [91]:
test_cases = [
    {"producto_id": "PROD-003", "cantidad": 2},
    {"producto_id": "PROD-015", "cantidad": 1},
    {"producto_id": "PROD-005", "cantidad": 3},
    {"producto_id": "PROD-010", "cantidad": 2},
    {"producto_id": "PROD-013", "cantidad": 10},
    {"producto_id": "PROD-016", "cantidad": 1},
    {"producto_id": "PROD-018", "cantidad": 4},
    {"producto_id": "PROD-001", "cantidad": 8},
]

rows = []
stock_df, pedidos_df, facturas_df = load_sheets()

for case in test_cases:
    pid = case["producto_id"]
    cant = case["cantidad"]
    row = stock_df[stock_df["producto_id"].astype(str).str.upper() == pid.upper()]
    if row.empty:
        continue
    r = row.iloc[0]
    stock = int(r["stock"])
    precio = float(r["precio_unitario"])
    total = cant * precio
    pr = fuzzy_priority(stock, cant, total)

    rows.append({
        "producto_id": pid,
        "cantidad": cant,
        "stock": stock,
        "precio_unitario": precio,
        "valor_total": total,
        "prioridad_score": pr["prioridad_score"],
        "prioridad_label": pr["prioridad_label"]
    })

eval_df = pd.DataFrame(rows)
eval_df

,producto_id,cantidad,stock,precio_unitario,valor_total,prioridad_score,prioridad_label
0,PROD-003,2,24,32000.0,64000.0,24.56,BAJA
1,PROD-015,1,5,890000.0,890000.0,50.00,MEDIA
2,PROD-005,3,6,420000.0,1260000.0,50.00,MEDIA
3,PROD-010,2,8,310000.0,620000.0,50.00,MEDIA
4,PROD-013,10,150,4200.0,42000.0,50.00,MEDIA
5,PROD-016,1,0,9200.0,9200.0,13.33,BAJA
6,PROD-018,4,0,14500.0,58000.0,13.38,BAJA
7,PROD-001,8,120,18000.0,144000.0,50.00,MEDIA


In [92]:
metricas_n3 = {
    "casos_evaluados": int(len(eval_df)),
    "prioridad_alta": int((eval_df["prioridad_label"] == "ALTA").sum()),
    "prioridad_media": int((eval_df["prioridad_label"] == "MEDIA").sum()),
    "prioridad_baja": int((eval_df["prioridad_label"] == "BAJA").sum()),
    "score_promedio": round(float(eval_df["prioridad_score"].mean()), 2) if len(eval_df) else 0.0
}

metricas_n3

{'casos_evaluados': 8,
 'prioridad_alta': 0,
 'prioridad_media': 5,
 'prioridad_baja': 3,
 'score_promedio': 37.66}

## 8. Pruebas locales del agente

In [ ]:
parsed, answer = process_message("quiero pedir PROD-003 cantidad 2")
print(parsed)
print("---")
print(answer)

In [ ]:
parsed, answer = process_message("consultar estado de mi ultimo pedido")
print(parsed)
print("---")
print(answer)

## 9. Integración con Telegram Bot

El tutorial del curso ya muestra que el token del bot se crea con BotFather y luego se valida con `python-telegram-bot`; aquí esa misma credencial se usa para conectar Telegram al agente programático.

In [65]:
from telegram import Update
from telegram.ext import ApplicationBuilder, CommandHandler, ContextTypes, MessageHandler, filters

async def start_command(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text(
        "Hola, soy tu asistente de logística. Puedes escribir: quiero pedir PROD-003 cantidad 2"
    )

async def handle_message(update: Update, context: ContextTypes.DEFAULT_TYPE):
    if not update.message or not update.message.text:
        return

    mensaje = update.message.text
    cliente = update.effective_user.first_name or "Cliente"
    chat_id = str(update.effective_chat.id)

    try:
        parsed, answer = process_message(mensaje=mensaje, cliente=cliente, chat_id=chat_id)
        print("\n===== TRAZA DEL AGENTE =====")
        print("Mensaje:", mensaje)
        print("Estructura detectada:", parsed.model_dump())
        print("Respuesta final:", answer)
        await update.message.reply_text(answer)
    except Exception as e:
        await update.message.reply_text(f"Ocurrió un error procesando tu solicitud: {e}")


def build_telegram_app():
    app = ApplicationBuilder().token(TELEGRAM_BOT_TOKEN).build()
    app.add_handler(CommandHandler("start", start_command))
    app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle_message))
    return app

## 10. Ejecutar el bot

Descomenta la última línea para dejar el bot corriendo en Colab. Si van a grabar video, esta es la parte ideal para mostrar la ejecución en tiempo real y la traza del agente.

In [70]:
app = build_telegram_app()
await app.initialize()
await app.start()
await app.updater.start_polling()

print("✅ Bot corriendo. Ahora envíale un mensaje desde Telegram.")

✅ Bot corriendo. Ahora envíale un mensaje desde Telegram.


In [67]:
await app.updater.stop()
await app.stop()
await app.shutdown()

print("🛑 Bot detenido.")

🛑 Bot detenido.


## 11. Ejemplo de evidencias para diapositivas 07–09

### Arquitectura
Telegram -> Handler Python -> ChatPromptTemplate -> Gemini -> Tools (Stock/Pedidos/Facturas) -> Respuesta al usuario.

### Tools implementadas
- `get_stock(producto_id)`
- `create_order(cliente, chat_id, producto_id, cantidad)`
- `get_order_status(chat_id, order_id)`

### Justificación técnica
- Gemini interpreta intención y extrae entidades.
- Las rules de negocio se ejecutan con tools determinísticas.
